In [2]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# Load GHSL SMOD dataset
image = ee.Image("JRC/GHSL/P2023A/GHS_SMOD_V2-0/2030")
smod = image.select("smod_code")

# Panama boundary
panama = (
    ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    .filter(ee.Filter.eq("country_na", "Panama"))
    .geometry()
)

smod = smod.clip(panama)

# Urban / Rural masks
urban = smod.gte(21).And(smod.lte(30))
rural = smod.gte(11).And(smod.lte(13))

urban_img = urban.unmask(0).toByte()
rural_img = rural.unmask(0).toByte()

# Distance surfaces
distance_to_urban = urban_img.distance(
    ee.Kernel.euclidean(50000, "meters")
).clip(panama)

distance_to_rural = rural_img.distance(
    ee.Kernel.euclidean(50000, "meters")
).clip(panama)

# Visualization
urban_distance_vis = {
    "min": 0,
    "max": 25000,
    "palette": [
        "#fff5eb",
        "#fd8d3c",
        "#d94801"
    ]
}

rural_distance_vis = {
    "min": 0,
    "max": 25000,
    "palette": [
        "#f7fbff",
        "#6baed6",
        "#08306b"
    ]
}

urban_mask_vis = {
    "palette": ["#e31a1c"]
}

rural_mask_vis = {
    "palette": ["#1f78b4"]
}

# Create map
Map = geemap.Map()

Map.centerObject(panama, 7)

# Basemap
# Map.add_basemap("CartoDB.Positron")

# Distance layers
Map.addLayer(
    distance_to_urban,
    urban_distance_vis,
    "Distance to Urban",
    True,
    0.8
)

Map.addLayer(
    distance_to_rural,
    rural_distance_vis,
    "Distance to Rural",
    False,
    0.8
)

# Reference masks
Map.addLayer(
    urban.selfMask(),
    urban_mask_vis,
    "Urban Areas",
    True
)

Map.addLayer(
    rural.selfMask(),
    rural_mask_vis,
    "Rural Areas",
    False
)

# Layer control
Map.addLayerControl()

Map

Map(center=[8.513881071215563, -80.10553238925849], controls=(WidgetControl(options=['position', 'transparent_…